# Prompt Engineering Lab

This lab walks through prompt engineering techniques using the Groq API.

**Labs covered:**
1. The Chat Completion API and its parameters (model, temperature, top-p, system / user messages)
2. Zero-shot vs Few-shot prompting
3. Chain of Verification
4. Reasoning vs non-reasoning models
5. Ensemble approach across models

All labs use the **same customer complaint** so you can compare techniques head to head.

Every API call below prints both the **prompt sent to the model** and the **response received**, so you can see exactly what the model saw and produced.

## Setup

In [1]:
!pip -q install groq


[notice] A new release of pip is available: 24.1.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import time
from collections import Counter
from getpass import getpass
from groq import Groq

In [8]:
os.environ['GROQ_API_KEY'] = getpass('GROQ_API_KEY: ')

In [9]:
client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

### Configurable models

All labs reference these variables instead of hard-coding model names. Change them here once and every cell below will pick up the new values.

In [10]:
PRIMARY_MODEL   = 'llama-3.3-70b-versatile'   # large general-purpose model
SECONDARY_MODEL = 'llama-3.1-8b-instant'      # smaller / faster model
REASONING_MODEL = 'qwen/qwen3-32b'             # reasoning model (exposes thoughts)

### The complaint we will classify

Every lab below reuses this same complaint, so you can directly compare the effect of each prompting technique.

In [11]:
complaint = (
    "The delivery guy just left the package at the gate without ringing the bell. "
    "I found it two hours later completely soaked from the rain."
)

categories = ['Delivery issue', 'Product quality', 'Customer service', 'Other']

---
## Lab 1 — The Chat Completion API and its parameters

The Groq Chat Completions endpoint accepts a list of messages and a few sampling parameters. The two most important sampling controls are:

- **`temperature`** – scales the logits before sampling. `0.0` is (nearly) deterministic; higher values (e.g. `1.5`) make output more diverse.
- **`top_p`** (nucleus sampling) – restricts sampling to the smallest set of tokens whose cumulative probability is `≥ top_p`. Lower values (e.g. `0.1`) keep only the most likely tokens.

Messages are role-tagged:
- **`system`** – instructions about *who* the model is and *how* it should behave.
- **`user`** – the actual request / data.

We will call the API directly (no helper functions) so every parameter is visible.

### 1.1 – A minimal call with only a user message

In [12]:
user_message = f'Classify this customer complaint into one of {categories}.\n\nComplaint: {complaint}'

print('========== Prompt (user) ==========')
print(user_message)

response = client.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {'role': 'user', 'content': user_message},
    ],
)

print('\n========== Response ==========')
print(response.choices[0].message.content)

========== Prompt (user) ==========
Classify this customer complaint into one of ['Delivery issue', 'Product quality', 'Customer service', 'Other'].

Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response ==========
The customer complaint can be classified as: 'Delivery issue'. 

The complaint specifically mentions an issue with how the package was delivered (left at the gate without notification and exposed to the rain), which falls under the category of delivery issues.


### 1.2 – Adding system instructions

A system message lets us pin the model's role and output format up-front, instead of repeating those instructions in every user turn.

In [13]:
system_instruction = (
    'You are a customer-support triage assistant. '
    'Classify each complaint into exactly one of the following categories: '
    f'{", ".join(categories)}. '
    'Respond with only the category name — no explanation, no punctuation.'
)

user_message = f'Complaint: {complaint}'

print('========== Prompt (system) ==========')
print(system_instruction)
print('\n========== Prompt (user) ==========')
print(user_message)

response = client.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {'role': 'system', 'content': system_instruction},
        {'role': 'user',   'content': user_message},
    ],
)

print('\n========== Response ==========')
print(response.choices[0].message.content)

========== Prompt (system) ==========
You are a customer-support triage assistant. Classify each complaint into exactly one of the following categories: Delivery issue, Product quality, Customer service, Other. Respond with only the category name — no explanation, no punctuation.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response ==========
Delivery issue


### 1.3 – The `temperature` parameter

Run the same prompt twice with different temperatures and compare. With `temperature=0.0` you should see a stable, terse answer; with a high temperature the wording becomes more varied.

In [14]:
user_message = f'Complaint: {complaint}'

print('========== Prompt (system) ==========')
print(system_instruction)
print('\n========== Prompt (user) ==========')
print(user_message)

for temp in [0.0, 1.5]:
    response = client.chat.completions.create(
        model=PRIMARY_MODEL,
        messages=[
            {'role': 'system', 'content': system_instruction},
            {'role': 'user',   'content': user_message},
        ],
        temperature=temp,
    )
    print(f'\n========== Response (temperature={temp}) ==========')
    print(response.choices[0].message.content)

========== Prompt (system) ==========
You are a customer-support triage assistant. Classify each complaint into exactly one of the following categories: Delivery issue, Product quality, Customer service, Other. Respond with only the category name — no explanation, no punctuation.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response (temperature=0.0) ==========
Delivery issue

========== Response (temperature=1.5) ==========
Delivery issue


### 1.4 – The `top_p` parameter (nucleus sampling)

Holding temperature constant, vary `top_p`. A small `top_p` (e.g. `0.1`) constrains the model to high-probability tokens; a value close to `1.0` allows the full distribution.

In [15]:
user_message = f'Complaint: {complaint}'

print('========== Prompt (system) ==========')
print(system_instruction)
print('\n========== Prompt (user) ==========')
print(user_message)

for top_p in [0.1, 1.0]:
    response = client.chat.completions.create(
        model=PRIMARY_MODEL,
        messages=[
            {'role': 'system', 'content': system_instruction},
            {'role': 'user',   'content': user_message},
        ],
        temperature=0.7,
        top_p=top_p,
    )
    print(f'\n========== Response (top_p={top_p}) ==========')
    print(response.choices[0].message.content)

========== Prompt (system) ==========
You are a customer-support triage assistant. Classify each complaint into exactly one of the following categories: Delivery issue, Product quality, Customer service, Other. Respond with only the category name — no explanation, no punctuation.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response (top_p=0.1) ==========
Delivery issue

========== Response (top_p=1.0) ==========
Delivery issue


---
## Lab 2 — Zero-shot vs Few-shot Prompting

- **Zero-shot:** describe the task; provide no examples.
- **Few-shot:** include a handful of (input, label) pairs so the model can pattern-match.

Few-shot is especially useful when the label set has nuanced boundaries (e.g. distinguishing *Delivery issue* from *Customer service*).

### 2.1 – Zero-shot classification

In [16]:
zero_shot_system = (
    'You are a customer-support triage assistant. '
    f'Classify the complaint into exactly one of: {", ".join(categories)}. '
    'Reply with only the category name.'
)

zero_shot_user = f'Complaint: {complaint}\nCategory:'

print('========== Prompt (system) ==========')
print(zero_shot_system)
print('\n========== Prompt (user) ==========')
print(zero_shot_user)

response = client.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {'role': 'system', 'content': zero_shot_system},
        {'role': 'user',   'content': zero_shot_user},
    ],
    temperature=0.0,
)

print('\n========== Response ==========')
print(response.choices[0].message.content)

========== Prompt (system) ==========
You are a customer-support triage assistant. Classify the complaint into exactly one of: Delivery issue, Product quality, Customer service, Other. Reply with only the category name.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.
Category:

========== Response ==========
Delivery issue


### 2.2 – Few-shot classification

We embed labelled examples directly in the user message so the model can infer the labelling convention.

In [17]:
few_shot_user = f"""Classify the complaint into exactly one of: {', '.join(categories)}.
Reply with only the category name.

Examples:

Complaint: \"My order was supposed to arrive yesterday, but it hasn't shown up yet.\"
Category: Delivery issue

Complaint: \"The shoes I received had a tear on the side.\"
Category: Product quality

Complaint: \"The support agent was rude and didn't help at all.\"
Category: Customer service

Complaint: \"I had trouble applying the discount code during checkout.\"
Category: Other

Now classify the following complaint:

Complaint: {complaint}
Category:"""

print('========== Prompt (user) ==========')
print(few_shot_user)

response = client.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {'role': 'user', 'content': few_shot_user},
    ],
    temperature=0.0,
)

print('\n========== Response ==========')
print(response.choices[0].message.content)

========== Prompt (user) ==========
Classify the complaint into exactly one of: Delivery issue, Product quality, Customer service, Other.
Reply with only the category name.

Examples:

Complaint: "My order was supposed to arrive yesterday, but it hasn't shown up yet."
Category: Delivery issue

Complaint: "The shoes I received had a tear on the side."
Category: Product quality

Complaint: "The support agent was rude and didn't help at all."
Category: Customer service

Complaint: "I had trouble applying the discount code during checkout."
Category: Other

Now classify the following complaint:

Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.
Category:

========== Response ==========
Delivery issue


---
## Lab 3 — Chain of Verification (CoVe)

Chain of Verification is a three-step prompting pattern that reduces hasty or under-evidenced conclusions:

1. **Generate** verification questions that probe the key facts in the input.
2. **Answer** each question using only what the input actually says.
3. **Classify** based on the verified answers, not the raw text.

We will run all three steps explicitly so the intermediate artefacts are visible.

### 3.1 – Step 1: generate verification questions

In [18]:
step1_system = (
    'You are an analyst preparing to classify a customer complaint. '
    'Generate exactly 3 short, fact-seeking verification questions whose answers '
    'would help decide which of these categories applies: '
    f'{", ".join(categories)}. '
    'Return the questions as a numbered list, nothing else.'
)
step1_user = f'Complaint: {complaint}'

print('========== Prompt (system) ==========')
print(step1_system)
print('\n========== Prompt (user) ==========')
print(step1_user)

response = client.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {'role': 'system', 'content': step1_system},
        {'role': 'user',   'content': step1_user},
    ],
    temperature=0.0,
)

verification_questions = response.choices[0].message.content
print('\n========== Response ==========')
print(verification_questions)

========== Prompt (system) ==========
You are an analyst preparing to classify a customer complaint. Generate exactly 3 short, fact-seeking verification questions whose answers would help decide which of these categories applies: Delivery issue, Product quality, Customer service, Other. Return the questions as a numbered list, nothing else.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response ==========
1. Was the package damaged as a result of being left outside in the rain?
2. Did the customer have any prior interactions with the customer service team regarding the delivery?
3. Was the issue with the delivery itself, such as the timing or the condition of the package when it was left?


### 3.2 – Step 2: answer the verification questions

We force the model to ground each answer in the complaint text. If the complaint does not contain the answer, the model must say so explicitly.

In [19]:
step2_system = (
    'Answer each verification question using ONLY the information in the complaint. '
    'If the complaint does not state the answer, reply \"Not specified\". '
    'Return one answer per question, in the same numbered order.'
)
step2_user = (
    f'Complaint: {complaint}\n\n'
    f'Questions:\n{verification_questions}'
)

print('========== Prompt (system) ==========')
print(step2_system)
print('\n========== Prompt (user) ==========')
print(step2_user)

response = client.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {'role': 'system', 'content': step2_system},
        {'role': 'user',   'content': step2_user},
    ],
    temperature=0.0,
)

verification_answers = response.choices[0].message.content
print('\n========== Response ==========')
print(verification_answers)

========== Prompt (system) ==========
Answer each verification question using ONLY the information in the complaint. If the complaint does not state the answer, reply "Not specified". Return one answer per question, in the same numbered order.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

Questions:
1. Was the package damaged as a result of being left outside in the rain?
2. Did the customer have any prior interactions with the customer service team regarding the delivery?
3. Was the issue with the delivery itself, such as the timing or the condition of the package when it was left?

========== Response ==========
1. Yes, the package was damaged as a result of being left outside in the rain, as it was found to be "completely soaked".
2. Not specified
3. Yes, the issue was with the delivery itself, specifically that the package was left at the gate 

### 3.3 – Step 3: classify using the verified findings

In [20]:
step3_system = (
    'You are a customer-support triage assistant. '
    f'Using the verified findings, classify the complaint into exactly one of: {", ".join(categories)}. '
    'Return JSON of the form {\"category\": \"...\", \"justification\": \"...\"} and nothing else.'
)
step3_user = (
    f'Complaint: {complaint}\n\n'
    f'Verification questions:\n{verification_questions}\n\n'
    f'Verified answers:\n{verification_answers}'
)

print('========== Prompt (system) ==========')
print(step3_system)
print('\n========== Prompt (user) ==========')
print(step3_user)

response = client.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {'role': 'system', 'content': step3_system},
        {'role': 'user',   'content': step3_user},
    ],
    temperature=0.0,
    response_format={'type': 'json_object'},
)

final_raw = response.choices[0].message.content
print('\n========== Response (raw) ==========')
print(final_raw)

final = json.loads(final_raw)
print('\n========== Response (parsed) ==========')
print(final)

========== Prompt (system) ==========
You are a customer-support triage assistant. Using the verified findings, classify the complaint into exactly one of: Delivery issue, Product quality, Customer service, Other. Return JSON of the form {"category": "...", "justification": "..."} and nothing else.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

Verification questions:
1. Was the package damaged as a result of being left outside in the rain?
2. Did the customer have any prior interactions with the customer service team regarding the delivery?
3. Was the issue with the delivery itself, such as the timing or the condition of the package when it was left?

Verified answers:
1. Yes, the package was damaged as a result of being left outside in the rain, as it was found to be "completely soaked".
2. Not specified
3. Yes, the issue was with the delivery its

---
## Lab 4 — Reasoning vs Non-reasoning Models

**Non-reasoning** models (e.g. Llama 3.3 70B) emit the answer directly.

**Reasoning** models (e.g. Qwen 3) produce an internal *chain of thought* before the answer. Groq exposes this via the `reasoning` field on the response message when you set `reasoning_format='parsed'`.

We send the same prompt to both and compare.

### 4.1 – Non-reasoning model

In [21]:
classify_system = (
    'You are a customer-support triage assistant. '
    f'Classify the complaint into exactly one of: {", ".join(categories)}. '
    'Return JSON of the form {\"category\": \"...\"} and nothing else.'
)
classify_user = f'Complaint: {complaint}'

print('========== Prompt (system) ==========')
print(classify_system)
print('\n========== Prompt (user) ==========')
print(classify_user)

response = client.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {'role': 'system', 'content': classify_system},
        {'role': 'user',   'content': classify_user},
    ],
    temperature=0.0,
    response_format={'type': 'json_object'},
)

print('\n========== Response ==========')
print(response.choices[0].message.content)

========== Prompt (system) ==========
You are a customer-support triage assistant. Classify the complaint into exactly one of: Delivery issue, Product quality, Customer service, Other. Return JSON of the form {"category": "..."} and nothing else.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response ==========
{
  "category": "Delivery issue"
}


### 4.2 – Reasoning model (Qwen)

`reasoning_format='parsed'` separates the model's thought trace from its final answer. The thoughts come back in `message.reasoning`, the answer in `message.content`.

In [22]:
print('========== Prompt (system) ==========')
print(classify_system)
print('\n========== Prompt (user) ==========')
print(classify_user)

response = client.chat.completions.create(
    model=REASONING_MODEL,
    messages=[
        {'role': 'system', 'content': classify_system},
        {'role': 'user',   'content': classify_user},
    ],
    temperature=0.0,
    reasoning_format='parsed',
    response_format={'type': 'json_object'},
)

msg = response.choices[0].message

print('\n========== Response (reasoning / thoughts) ==========')
print(getattr(msg, 'reasoning', '(none returned)'))

print('\n========== Response (final answer) ==========')
print(msg.content)

========== Prompt (system) ==========
You are a customer-support triage assistant. Classify the complaint into exactly one of: Delivery issue, Product quality, Customer service, Other. Return JSON of the form {"category": "..."} and nothing else.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response (reasoning / thoughts) ==========
Okay, let's see. The user is complaining about their package being left at the gate without ringing the bell and getting soaked in the rain. Hmm. So first, the delivery person didn't ring the bell, which might mean they didn't follow the usual procedure. Then the package was left outside and got wet. 

Wait, the main issue here is about how the delivery was handled. The user is upset that the delivery guy didn't notify them and left the package outside, leading to damage. So the categories to consider are De

---
## Lab 5 — Ensemble Approach

An **ensemble** asks several different models the same question and aggregates their answers (here, by majority vote). This usually beats any single model when the models make uncorrelated mistakes.

We will query three models — two Llama variants and one Qwen — then aggregate.

In [23]:
ensemble_system = (
    'You are a customer-support triage assistant. '
    f'Classify the complaint into exactly one of: {", ".join(categories)}. '
    'Return JSON of the form {\"category\": \"...\"} and nothing else.'
)
ensemble_user = f'Complaint: {complaint}'

print('========== Prompt (system) ==========')
print(ensemble_system)
print('\n========== Prompt (user) ==========')
print(ensemble_user)

ensemble_models = [PRIMARY_MODEL, SECONDARY_MODEL, REASONING_MODEL]
predictions = []

for model in ensemble_models:
    kwargs = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': ensemble_system},
            {'role': 'user',   'content': ensemble_user},
        ],
        'temperature': 0.0,
        'response_format': {'type': 'json_object'},
    }
    if model == REASONING_MODEL:
        kwargs['reasoning_format'] = 'parsed'

    response = client.chat.completions.create(**kwargs)
    raw = response.choices[0].message.content
    category = json.loads(raw)['category']
    predictions.append((model, category))

    print(f'\n========== Response from {model} ==========')
    print('Raw     :', raw)
    print('Category:', category)
    time.sleep(1)  # be gentle with rate limits

========== Prompt (system) ==========
You are a customer-support triage assistant. Classify the complaint into exactly one of: Delivery issue, Product quality, Customer service, Other. Return JSON of the form {"category": "..."} and nothing else.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response from llama-3.3-70b-versatile ==========
Raw     : {
  "category": "Delivery issue"
}
Category: Delivery issue

========== Response from llama-3.1-8b-instant ==========
Raw     : {
  "category": "Delivery issue"
}
Category: Delivery issue

========== Response from qwen/qwen3-32b ==========
Raw     : {"category": "Delivery issue"}
Category: Delivery issue


In [24]:
votes = Counter(category for _, category in predictions)
majority_category, majority_count = votes.most_common(1)[0]

print('Per-model predictions:')
for model, category in predictions:
    print(f'  {model:35s} -> {category}')

print('\nVotes        :', dict(votes))
print('Majority vote:', majority_category, f'({majority_count}/{len(predictions)})')

Per-model predictions:
  llama-3.3-70b-versatile             -> Delivery issue
  llama-3.1-8b-instant                -> Delivery issue
  qwen/qwen3-32b                      -> Delivery issue

Votes        : {'Delivery issue': 3}
Majority vote: Delivery issue (3/3)
